# Active Speaker Detection

Réplica de la demo oficial de **TalkNet** ([`demoTalkNet.py`](https://github.com/TaoRuijie/TalkNet-ASD/blob/main/demoTalkNet.py), repo [TaoRuijie/TalkNet-ASD](https://github.com/TaoRuijie/TalkNet-ASD), licencia MIT; paper: [*Is Someone Speaking?*, ACM MM 2021](https://arxiv.org/abs/2107.06592)). TalkNet cruza el movimiento de los labios de cada cara con el audio y da, por fotograma, una puntuación: **> 0 = esa cara está hablando**.

Pasos de la demo: vídeo a 25 fps → cortes de plano → detección de caras → seguimiento de caras → recorte de cada cara con su audio → TalkNet. Las funciones se copian de la demo con su mismo nombre; **se reciclan dos piezas del baseline**: los cortes de plano (PySceneDetect) y el detector de caras de InsightFace (en lugar de S3FD).

**Resultado:** la tabla `asd` (una fila por cara y fotograma) y la función `añadir_talking(df, asd, ...)`, que añade la columna `talking` a los *tracks* del baseline.

## 0. Configuración

Código y modelo de TalkNet. Usamos el modelo entrenado en *TalkSet* (vídeos "en la vida real"): los autores avisan en su [FAQ](https://github.com/TaoRuijie/TalkNet-ASD/blob/main/FAQ.md) de que el entrenado en AVA generaliza mal fuera de AVA.

In [12]:
import os, sys, glob, math, subprocess
import cv2, numpy, torch, tqdm, python_speech_features, gdown
import pandas as pd
from scipy import signal
from scipy.io import wavfile
from scipy.interpolate import interp1d

VIDEO = "clip_quanta.mp4"     # <- el mismo vídeo que en el baseline

# Código y modelo de TalkNet (solo la primera vez): https://github.com/TaoRuijie/TalkNet-ASD
if not os.path.isdir("TalkNet-ASD"):
    subprocess.run("git clone -q --depth 1 https://github.com/TaoRuijie/TalkNet-ASD.git", shell=True, check=True)
pretrainModel = "TalkNet-ASD/pretrain_TalkSet.model"   # modelo para vídeos "en la vida real" (el de AVA generaliza peor)
if not os.path.isfile(pretrainModel):
    gdown.download(id="1AbN9fCf9IexMxEKXLQY2KYBlb-IhSEea", output=pretrainModel)   # mismo id que demoTalkNet.py
sys.path.insert(0, "TalkNet-ASD")
from talkNet import talkNet

# Carpetas de trabajo y parámetros por defecto de demoTalkNet.py
savePath = os.path.join("asd_work", os.path.splitext(os.path.basename(VIDEO))[0])
pyaviPath, pyframesPath, pycropPath = (os.path.join(savePath, d) for d in ("pyavi", "pyframes", "pycrop"))
for p in (pyaviPath, pyframesPath, pycropPath):
    os.makedirs(p, exist_ok=True)
minTrack, numFailedDet, minFaceSize, cropScale, nDataLoaderThread = 10, 10, 1, 0.40, 10

## 1. Vídeo a 25 fps, audio y fotogramas

TalkNet trabaja a 25 fps y con audio a 16 kHz.

In [13]:
# Mismos comandos ffmpeg que main() en demoTalkNet.py: vídeo a 25 fps, audio mono a 16 kHz y fotogramas en .jpg
videoFilePath = os.path.join(pyaviPath, "video.avi")
audioFilePath = os.path.join(pyaviPath, "audio.wav")
subprocess.call("ffmpeg -y -i %s -qscale:v 2 -threads %d -async 1 -r 25 %s -loglevel panic" % (VIDEO, nDataLoaderThread, videoFilePath), shell=True)
subprocess.call("ffmpeg -y -i %s -qscale:a 0 -ac 1 -vn -threads %d -ar 16000 %s -loglevel panic" % (videoFilePath, nDataLoaderThread, audioFilePath), shell=True)
subprocess.call("ffmpeg -y -i %s -qscale:v 2 -threads %d -f image2 %s -loglevel panic" % (videoFilePath, nDataLoaderThread, os.path.join(pyframesPath, "%06d.jpg")), shell=True)
flist = sorted(glob.glob(os.path.join(pyframesPath, "*.jpg")))
print(len(flist), "fotogramas a 25 fps")

2401 fotogramas a 25 fps


## 2. Cortes de plano (reciclado del baseline)

In [14]:
# Reciclado del baseline (sección 1.1): mismo detector de cortes de PySceneDetect.
# demoTalkNet.py usa la API antigua (VideoManager), que ya no existe en las versiones actuales.
from scenedetect import detect, ContentDetector

scene = [(s[0].frame_num, s[1].frame_num) for s in detect(videoFilePath, ContentDetector())]
if not scene:                                 # sin cortes: todo el vídeo es un solo plano
    scene = [(0, len(flist))]
print(len(scene), "planos")

7 planos


## 3. Detección de caras (reciclado del baseline: InsightFace)

In [15]:
# Reciclado del baseline (sección 2): el detector de caras de InsightFace en lugar de S3FD.
# Mismo formato de salida que inference_video() en demoTalkNet.py: una lista por fotograma con {'frame', 'bbox', 'conf'}
from insightface.app import FaceAnalysis

app = FaceAnalysis(name="buffalo_l", allowed_modules=["detection"], providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))

faces = []
for fidx, fname in enumerate(tqdm.tqdm(flist)):
    faces.append([{"frame": fidx, "bbox": f.bbox.tolist(), "conf": float(f.det_score)} for f in app.get(cv2.imread(fname))])

/home/rjvillen/projects/attention-is-all-she-needs/.venv/lib/python3.13/site-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
100%|██████████| 2401/2401 [03:46<00:00, 10.60it/s]


## 4. Seguimiento de caras

Dentro de cada plano, une las detecciones de fotogramas consecutivos que se solapan (IoU > 0,5) y rellena los huecos interpolando.

In [16]:
# Copiado de demoTalkNet.py: bb_intersection_over_union() y track_shot() (args.* -> variables de la configuración)
def bb_intersection_over_union(boxA, boxB):
	xA = max(boxA[0], boxB[0])
	yA = max(boxA[1], boxB[1])
	xB = min(boxA[2], boxB[2])
	yB = min(boxA[3], boxB[3])
	interArea = max(0, xB - xA) * max(0, yB - yA)
	boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
	boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
	return interArea / float(boxAArea + boxBArea - interArea)

def track_shot(sceneFaces):
	iouThres  = 0.5     # Minimum IOU between consecutive face detections
	tracks    = []
	while True:
		track     = []
		for frameFaces in sceneFaces:
			for face in frameFaces:
				if track == []:
					track.append(face)
					frameFaces.remove(face)
				elif face['frame'] - track[-1]['frame'] <= numFailedDet:
					iou = bb_intersection_over_union(face['bbox'], track[-1]['bbox'])
					if iou > iouThres:
						track.append(face)
						frameFaces.remove(face)
						continue
				else:
					break
		if track == []:
			break
		elif len(track) > minTrack:
			frameNum    = numpy.array([ f['frame'] for f in track ])
			bboxes      = numpy.array([numpy.array(f['bbox']) for f in track])
			frameI      = numpy.arange(frameNum[0],frameNum[-1]+1)
			bboxesI    = []
			for ij in range(0,4):
				interpfn  = interp1d(frameNum, bboxes[:,ij])
				bboxesI.append(interpfn(frameI))
			bboxesI  = numpy.stack(bboxesI, axis=1)
			if max(numpy.mean(bboxesI[:,2]-bboxesI[:,0]), numpy.mean(bboxesI[:,3]-bboxesI[:,1])) > minFaceSize:
				tracks.append({'frame':frameI,'bbox':bboxesI})
	return tracks

# Como en main() de demoTalkNet.py: se sigue cada plano por separado y se descartan los planos muy cortos
allTracks = []
for shot in scene:
	if shot[1] - shot[0] >= minTrack:
		allTracks.extend(track_shot(faces[shot[0]:shot[1]]))
print(len(allTracks), "pistas de caras")

34 pistas de caras


## 5. Recortar cada cara con su audio

In [17]:
# Copiado de demoTalkNet.py: crop_video() recorta cada pista de cara (224x224) con su trozo de audio
def crop_video(track, cropFile):
	vOut = cv2.VideoWriter(cropFile + 't.avi', cv2.VideoWriter_fourcc(*'XVID'), 25, (224,224))# Write video
	dets = {'x':[], 'y':[], 's':[]}
	for det in track['bbox']: # Read the tracks
		dets['s'].append(max((det[3]-det[1]), (det[2]-det[0]))/2)
		dets['y'].append((det[1]+det[3])/2) # crop center x
		dets['x'].append((det[0]+det[2])/2) # crop center y
	dets['s'] = signal.medfilt(dets['s'], kernel_size=13)  # Smooth detections
	dets['x'] = signal.medfilt(dets['x'], kernel_size=13)
	dets['y'] = signal.medfilt(dets['y'], kernel_size=13)
	for fidx, frame in enumerate(track['frame']):
		cs  = cropScale
		bs  = dets['s'][fidx]   # Detection box size
		bsi = int(bs * (1 + 2 * cs))  # Pad videos by this amount
		image = cv2.imread(flist[frame])
		frame = numpy.pad(image, ((bsi,bsi), (bsi,bsi), (0, 0)), 'constant', constant_values=(110, 110))
		my  = dets['y'][fidx] + bsi  # BBox center Y
		mx  = dets['x'][fidx] + bsi  # BBox center X
		face = frame[int(my-bs):int(my+bs*(1+2*cs)),int(mx-bs*(1+cs)):int(mx+bs*(1+cs))]
		vOut.write(cv2.resize(face, (224, 224)))
	audioTmp    = cropFile + '.wav'
	audioStart  = (track['frame'][0]) / 25
	audioEnd    = (track['frame'][-1]+1) / 25
	vOut.release()
	command = ("ffmpeg -y -i %s -async 1 -ac 1 -vn -acodec pcm_s16le -ar 16000 -threads %d -ss %.3f -to %.3f %s -loglevel panic" % \
		      (audioFilePath, nDataLoaderThread, audioStart, audioEnd, audioTmp))
	output = subprocess.call(command, shell=True, stdout=None) # Crop audio file
	command = ("ffmpeg -y -i %st.avi -i %s -threads %d -c:v copy -c:a copy %s.avi -loglevel panic" % \
			  (cropFile, audioTmp, nDataLoaderThread, cropFile)) # Combine audio and video file
	output = subprocess.call(command, shell=True, stdout=None)
	os.remove(cropFile + 't.avi')
	return {'track':track, 'proc_track':dets}

vidTracks = [crop_video(track, os.path.join(pycropPath, '%05d' % ii)) for ii, track in enumerate(tqdm.tqdm(allTracks))]

100%|██████████| 34/34 [00:11<00:00,  2.85it/s]


## 6. TalkNet: puntuación por fotograma

Para cada pista, TalkNet recibe la secuencia de la boca (112×112, gris) y el audio (MFCC) y devuelve una puntuación por fotograma. La demo promedia varias longitudes de ventana (`durationSet`) para que sea más estable.

In [18]:
# Copiado de demoTalkNet.py: evaluate_network() da una puntuación por fotograma a cada pista (> 0: habla)
def evaluate_network(files):
	s = talkNet()
	s.loadParameters(pretrainModel)
	s.eval()
	allScores = []
	durationSet = {1,1,1,2,2,2,3,3,4,5,6} # Use this line can get more reliable result
	for file in tqdm.tqdm(files, total = len(files)):
		fileName = os.path.splitext(file.split('/')[-1])[0] # Load audio and video
		_, audio = wavfile.read(os.path.join(pycropPath, fileName + '.wav'))
		audioFeature = python_speech_features.mfcc(audio, 16000, numcep = 13, winlen = 0.025, winstep = 0.010)
		video = cv2.VideoCapture(os.path.join(pycropPath, fileName + '.avi'))
		videoFeature = []
		while video.isOpened():
			ret, frames = video.read()
			if ret == True:
				face = cv2.cvtColor(frames, cv2.COLOR_BGR2GRAY)
				face = cv2.resize(face, (224,224))
				face = face[int(112-(112/2)):int(112+(112/2)), int(112-(112/2)):int(112+(112/2))]
				videoFeature.append(face)
			else:
				break
		video.release()
		videoFeature = numpy.array(videoFeature)
		length = min((audioFeature.shape[0] - audioFeature.shape[0] % 4) / 100, videoFeature.shape[0] / 25)
		audioFeature = audioFeature[:int(round(length * 100)),:]
		videoFeature = videoFeature[:int(round(length * 25)),:,:]
		allScore = [] # Evaluation use TalkNet
		for duration in durationSet:
			batchSize = int(math.ceil(length / duration))
			scores = []
			with torch.no_grad():
				for i in range(batchSize):
					inputA = torch.FloatTensor(audioFeature[i * duration * 100:(i+1) * duration * 100,:]).unsqueeze(0).cuda()
					inputV = torch.FloatTensor(videoFeature[i * duration * 25: (i+1) * duration * 25,:,:]).unsqueeze(0).cuda()
					embedA = s.model.forward_audio_frontend(inputA)
					embedV = s.model.forward_visual_frontend(inputV)
					embedA, embedV = s.model.forward_cross_attention(embedA, embedV)
					out = s.model.forward_audio_visual_backend(embedA, embedV)
					score = s.lossAV.forward(out, labels = None)
					scores.extend(score)
			allScore.append(scores)
		allScore = numpy.round((numpy.mean(numpy.array(allScore), axis = 0)), 1).astype(float)
		allScores.append(allScore)
	return allScores

scores = evaluate_network(sorted(glob.glob("%s/*.avi" % pycropPath)))

10-22 21:05:48 Model para number = 15.01


100%|██████████| 34/34 [00:15<00:00,  2.17it/s]


## 7. Resultado: una fila por cara y fotograma

In [19]:
# Una fila por cara y fotograma (25 fps). Puntuación suavizada ±2 fotogramas y talking = score >= 0,
# igual que visualization() en demoTalkNet.py (caja verde = habla). La caja de la cara es la que dibuja la demo.
filas = []
for tidx, track in enumerate(vidTracks):
	score = scores[tidx]
	for fidx, frame in enumerate(track['track']['frame'].tolist()):
		s = numpy.mean(score[max(fidx - 2, 0): min(fidx + 3, len(score) - 1)])   # average smoothing
		x, y, half = track['proc_track']['x'][fidx], track['proc_track']['y'][fidx], track['proc_track']['s'][fidx]
		filas.append({"frame": frame, "t": frame / 25, "face_track": tidx, "x1": x - half, "y1": y - half,
		              "x2": x + half, "y2": y + half, "score": round(float(s), 2), "talking": bool(s >= 0)})
asd = pd.DataFrame(filas)
asd.to_csv(f"dataframes/asd_{os.path.basename(savePath)}.tsv", sep="\t", index=False)
print(f"{asd.face_track.nunique()} pistas de caras; hablando en el {100 * asd.talking.mean():.0f} % de las detecciones")
asd.head()

34 pistas de caras; hablando en el 42 % de las detecciones


,frame,t,face_track,x1,y1,x2,y2,score,talking
0,0,0.00,0,257.196362,76.665409,365.333241,184.802288,1.57,True
1,1,0.04,0,257.108669,76.922108,365.420933,185.234371,1.72,True
2,2,0.08,0,256.946941,77.279209,365.582661,185.914928,1.82,True
3,3,0.12,0,257.290142,77.279209,365.925861,185.914928,2.12,True
4,4,0.16,0,257.729351,77.279209,366.365070,185.914928,2.36,True


In [20]:
# Vídeo anotado como en visualization() de demoTalkNet.py: verde = habla, rojo = no habla
vOut = cv2.VideoWriter(os.path.join(pyaviPath, "video_only.avi"), cv2.VideoWriter_fourcc(*"XVID"), 25, cv2.imread(flist[0]).shape[1::-1])
por_frame = asd.groupby("frame")
for fidx, fname in enumerate(tqdm.tqdm(flist)):
	image = cv2.imread(fname)
	for r in (por_frame.get_group(fidx).itertuples() if fidx in por_frame.groups else []):
		clr = 255 if r.talking else 0
		cv2.rectangle(image, (int(r.x1), int(r.y1)), (int(r.x2), int(r.y2)), (0, clr, 255 - clr), 10)
		cv2.putText(image, "%s" % r.score, (int(r.x1), int(r.y1)), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, clr, 255 - clr), 5)
	vOut.write(image)
vOut.release()
# Con audio y en H.264 para verlo en el navegador
subprocess.call(f"ffmpeg -y -i {pyaviPath}/video_only.avi -i {audioFilePath} -c:v libx264 -pix_fmt yuv420p -c:a aac {pyaviPath}/video_out.mp4 -loglevel panic", shell=True)
from IPython.display import Video
Video(f"{pyaviPath}/video_out.mp4", embed=True, width=640)

100%|██████████| 2401/2401 [00:07<00:00, 323.80it/s]


## 8. Integración con el baseline

`añadir_talking()` añade a cada fila del `df` de *tracks* del baseline las columnas `talking` y `asd_score`, con la misma regla de siempre: la cara es de la persona cuya caja contiene su centro. A partir de ahí, `% de tiempo en pantalla hablando` por *track* o persona es una línea.

In [21]:
def añadir_talking(df, asd, fps, stride):
    """Añade a cada detección del baseline (df: frame, x1, y1, x2, y2) si esa persona está hablando.

    Misma regla que el paso 2 del baseline: la cara pertenece a la persona cuya caja contiene su centro.
    df está en fotogramas analizados (stride) del vídeo original; asd, en fotogramas a 25 fps: se emparejan por tiempo.
    """
    asd = asd.assign(cx=(asd.x1 + asd.x2) / 2, cy=(asd.y1 + asd.y2) / 2)
    por_frame = asd.groupby("frame")
    frame25 = ((df.frame * stride + stride - 1) / fps * 25).round().astype(int)   # segundo real -> fotograma a 25 fps
    talking, score = [], []
    for f25, r in zip(frame25, df.itertuples()):
        caras = por_frame.get_group(f25) if f25 in por_frame.groups else asd.iloc[:0]
        dentro = caras[(r.x1 <= caras.cx) & (caras.cx <= r.x2) & (r.y1 <= caras.cy) & (caras.cy <= r.y2)]
        talking.append(bool(dentro.talking.any()))
        score.append(dentro.score.max() if len(dentro) else numpy.nan)
    return df.assign(talking=talking, asd_score=score)

# Uso en el baseline, después de la sección 1.2 (df con los tracks de YOLO):
#   df = añadir_talking(df, pd.read_csv("dataframes/asd_robot_social_CSIC.tsv", sep="\t"), FPS, STRIDE)
#   per_track["pct_talking"] = 100 * df.groupby("track").talking.mean()      # % del tiempo en pantalla hablando